# Systems, Linux, and Networking Foundations

Maps to `design3.md` Phase 3.

This notebook is for removing black-box thinking around the operating system and the network.


## Core Topics

- process vs thread vs coroutine
- virtual memory, heap, stack, and page faults
- file descriptors and leak patterns
- buffered I/O and flush semantics
- shell and Linux debugging tools: `ps`, `top` or `htop`, `lsof`, `ss`, `strace`, `curl`
- concurrency (hands-on):
  - threading: locks, RLock, race conditions, deadlock prevention
  - multiprocessing: Pool, shared memory, IPC overhead
  - asyncio: event loop, coroutines, gather
  - producer-consumer pattern with `queue.Queue`
  - GIL interaction with each concurrency model
  - lock ordering as a deadlock prevention strategy
- networking fundamentals:
  - DNS resolution: what happens, caching, TTL
  - TCP: three-way handshake, connection pooling, keep-alive
  - HTTP request lifecycle: from DNS to response
  - TLS handshake at a conceptual level
  - common failure modes: connection refused, timeout, DNS failure, certificate expiry


## Drill Bank

- inspect open files and explain what can fail
- trace a request from DNS to HTTP response using `curl -v` and `ss`
- explain why buffered I/O can hide latency or durability issues
- use one Linux debugging tool to inspect process, FD, or network state
- write a short note on how memory pressure surfaces in real systems
- write a race condition, then fix it with a lock
- implement producer-consumer with `queue.Queue`
- compare threading vs asyncio vs sequential for I/O-bound work (e.g., 20 mock fetches)
- diagnose a simulated network failure (DNS, TCP, TLS, or HTTP level) and explain where it broke

## Artifact Target

- small debugging workbook
- short notes on process, memory, FD, and network behavior
- one small Linux debugging cheat sheet
- concurrency comparison benchmark (threads vs asyncio vs sequential)


In [ ]:
import queue
import threading
import time

counter = 0
counter_lock = threading.Lock()


def increment_without_lock(iterations):
    global counter
    for _ in range(iterations):
        counter += 1


def increment_with_lock(iterations):
    global counter
    for _ in range(iterations):
        with counter_lock:
            counter += 1


# TODO: run both versions from multiple threads and compare behavior.


In [ ]:
work_queue = queue.Queue()


def producer(items):
    for item in items:
        work_queue.put(item)
    work_queue.put(None)


def consumer(process):
    results = []
    while True:
        item = work_queue.get()
        if item is None:
            work_queue.put(None)
            break
        results.append(process(item))
        work_queue.task_done()
    return results


# TODO: use this skeleton to build a producer-consumer example.


In [ ]:
network_failure_map = {
    "connection refused": "TCP-level failure: target host reachable, but nothing accepting connections on that port.",
    "connection timed out": "Network or firewall path issue, or target host not responding.",
    "could not resolve host": "DNS failure before any TCP connection is attempted.",
    "certificate verify failed": "TLS handshake or trust-chain failure after TCP connection succeeds.",
    "502 bad gateway": "Application or upstream gateway/proxy path issue at HTTP layer.",
}

network_failure_map


## Exit Criteria

- I can explain common process, memory, and file-descriptor failures.
- I can reason about a basic network request path.
- I can use basic Linux tooling to inspect process, file, and network state.
- I can debug at least one level below the application code.
- I can explain when to use threads, processes, or asyncio and why.
- I can explain how the GIL interacts with each concurrency model.
- I can implement a producer-consumer pipeline.
- I can trace a network request from DNS to HTTP response and explain each layer.
- I can diagnose which network layer failed given symptoms.
